# MetaPulsar usage

Combine pulsar timing data from several PTA releases into one `MetaPulsar`.
The object is an Enterprise-compatible duck (`toas`, `residuals`, `Mmat`,
`fitpars`) and a `TimingPulsar` for linear and nonlinear timing. Sampler-facing
nonlinear timing lives in nltiming; this notebook stops at construction,
alignment, and export.

## Workflow

1. Build one pulsar by hand (`file_data` dict)
2. Align with `AlignmentPolicy` and write portable par/tim + feather
3. Inspect the Enterprise duck and parameter names
4. Discover layouts and build a small batch

## Defaults worth knowing

- `combination_strategy="shared"` — one astrophysical model; detector terms stay PTA-local
- `per_pta` — leave each release model untouched (not physical; diagnostic only)
- First PTA in `file_data` is the reference (values copied into the other legs)
- `exclude_from_shared=("DM",)` — each PTA keeps its own reference DM
- Merged fit parameters have no suffix; PTA-specific ones keep `_{pta}`

In [ ]:
import sys
from pathlib import Path

import loguru

from metapulsar import (
    AlignmentPolicy,
    PTA_DATA_RELEASES,
    combine_layouts,
    create_all_metapulsars,
    create_metapulsar,
    discover_files,
    discover_layout,
    filter_file_data_by_pulsars,
    get_pulsar_names_from_file_data,
    pta_summary,
)

loguru.logger.remove()
loguru.logger.add(sys.stdout, level="WARNING")

## Part 1: One pulsar by hand

`file_data` is a dict of PTA name → list of `{par, tim, timing_package}` records.
The first key is the reference PTA. `timing_package` is `"tempo2"` or `"pint"`.

J0613-0200 below is a mixed stack (EPTA/PPTA tempo2 + NANOGrav PINT). That is
the case that needs a common PINT/Tempo2 surface.

In [ ]:
pulsar_data = {
    "epta_dr2": [
        {
            "par": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par",
            "tim": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim",
            "timing_package": "tempo2",
        }
    ],
    "nanograv_9y": [
        {
            "par": "../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par",
            "tim": "../../data/ipta-dr2/NANOGrav_9y/tim/J0613-0200_NANOGrav_9yv1.tim",
            "timing_package": "pint",
        }
    ],
    "ppta_dr2": [
        {
            "par": "../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par",
            "tim": "../../data/ipta-dr2/PPTA_dr1dr2/tim/J0613-0200_dr1dr2.tim",
            "timing_package": "tempo2",
        }
    ],
}


### Alignment, binary models, and export

`shared` rewrites every PTA par onto one deterministic surface. `AlignmentPolicy`
is the only user-facing knob; the rest of the mixed-engine profile is fixed
because only that combination is validated for PINT↔Tempo2 residual parity.

- `unsupported="strip"` (default) drops families outside the common surface
  (DMMODEL, WaveX/IFUNC, glitches, …) with a warning. `"error"` fails instead.
- Pin `ephem` / `clock` / `bipm_version` / `ne_sw` when the reference par is
  ambiguous. A bare `TT(BIPM)` needs a year.
- `per_pta` plus a policy raises. Use `per_pta` only to keep release models
  byte-for-byte.

**Binary.** When `"binary"` is shared on a mixed PINT+Tempo2 stack, a plain
ELL1 / T2-EPS model may convert to `DD`, and a complete ELL1H block to `DDH`,
if the two-term scale gate exceeds 1 ns. Inspect
`mp.binary_conversion_report`. J0613-0200 is a low-e T2-EPS system, so the
report is typically `skip` / `below_threshold` — conversion did not fire.
H3-only ELL1H refuses by default (`h3_only="error"`); that path is not shown
here.

**Canonical tim.** `canonicalize_tim=False` (factory default) loads each release
`.tim` in place. `True` writes a standalone Tempo2 `FORMAT 1` file: INCLUDEs
flattened, `TIME` baked into MJDs, TOA names rewritten to `toaNNNNN`, and
`-pta` / `-pta_dataset` / `-timing_package` stamped on every TOA.

**Combination products.** `combination_output_dir` (requires `shared` +
`canonicalize_tim=True`) writes the portable interchange tree — not the
per-engine `parfile_output_dir` inputs:

```text
output/par/{pulsar}.par
output/tim/{pulsar}.tim
output/tim/{pulsar}/{pulsar}_{pta}.tim
```

The written par is meant to reload in **both** PINT and Tempo2. The rule is
the highest common *native* representation, not Tempo2's historical defaults:

- `UNITS TDB` — PINT is TDB-only; TCB→PINT is a partial conversion that
  leaves EQUAD/ECORR and GP noise untransformed
- `T2CMETHOD IAU2000B` — modern Earth orientation; do not emit `TEMPO`
- `TIMEEPH FB90`, explicit `EPHEM` and `CLK` (Tempo2 spelling; PINT accepts it)
- ICRS `RAJ` / `DECJ` when the reference is equatorial (J0613 is). If
  `ELONG`/`ELAT` are kept, write an explicit `ECL` — never leave the obliquity
  implicit (mixed stacks use IERS2003)

`T2CMETHOD TEMPO`, implicit ecliptic frames, and TCB are rejected for new
interchange files even when they are a "lowest common denominator."

In [ ]:
mp = create_metapulsar(
    file_data=pulsar_data,
    combination_strategy="shared",
    alignment_policy=AlignmentPolicy(),
    canonicalize_tim=True,
    combination_output_dir="./output",
)

print(mp.name)
print("PTAs:", list(pulsar_data))
print("strategy:", mp.combination_strategy)
print("merged components:", mp.combine_components)
print("TOAs:", len(mp.toas))
print(f"freq: {mp.freqs.min():.1f}–{mp.freqs.max():.1f} MHz")
print(f"span: {(mp.toas.max() - mp.toas.min()) / 86400:.1f} days")

In [ ]:
report = mp.binary_conversion_report
decision = report.decision
print(decision.outcome, decision.reason)
print("source:", decision.source_family, "→ target:", decision.target_family)
print("PINT resolved binary:", decision.resolved_binary_model)

### Portable par / tim / feather

`mp.combination_write_result` points at the files just written. The combination
par should carry the mixed-engine profile explicitly. Then write an
Enterprise/Discovery feather snapshot of the same pulsar.

In [ ]:
written = mp.combination_write_result
print(written.par_path)
print(written.tim_path)

keys = {
    "UNITS",
    "T2CMETHOD",
    "TIMEEPH",
    "EPHEM",
    "CLK",
    "CLOCK",
    "RAJ",
    "DECJ",
    "ECL",
    "BINARY",
}
for line in written.par_path.read_text().splitlines():
    parts = line.split()
    if parts and parts[0].upper() in keys:
        print(line)

print("--- output tree ---")
for path in sorted(Path("./output").rglob("*")):
    if path.is_file():
        print(path)

feather_path = Path("./output") / f"{mp.name}.feather"
mp.to_feather(feather_path)
print("feather:", feather_path)

## Part 2: Enterprise duck and parameter names

`MetaPulsar` is a fully usable Enterprise pulsar: combined TOAs, one
astrophysical model, PTA-suffixed detector terms. Merged parameters have no
suffix and inherit the reference PTA; PTA-specific parameters keep `_{pta}`.

In [ ]:
suffixes = tuple(f"_{pta}" for pta in pulsar_data)
merged = [p for p in mp.fitpars if not any(s in p for s in suffixes)]
local = [p for p in mp.fitpars if any(s in p for s in suffixes)]

print("merged (first 8):", merged[:8])
print("PTA-specific (first 8):", local[:8])
print("Mmat:", mp.Mmat.shape)

## Linear vs nonlinear timing (`TimingPulsar`)

`MetaPulsar` implements the `TimingPulsar` protocol.

- **Linear** — freeze the timing model at the par-file point and analytically
  marginalize. The Enterprise duck is that view: `residuals`, `toaerrs`, and
  `Mmat` (the PINT/tempo2 fitter design matrix, sign
  \(r(\theta+\delta)\approx r(\theta)-M\delta\)).
- **Nonlinear** — evaluate residuals at a new \(\theta\) through
  `mp.timing_engine(...)` or the convenience wrapper `mp.timing(...)`.

This notebook does not construct an engine. Sampler-facing nonlinear timing
(model config, Discovery/NumPyro, PTMCMC) lives in nltiming; see
[nonlinear_timing.ipynb](nonlinear_timing.ipynb).

In [ ]:
print("timing_engine:", callable(getattr(mp, "timing_engine", None)))
print("timing:", callable(getattr(mp, "timing", None)))
print("can_use_engines:", callable(getattr(mp, "can_use_engines", None)))

## Part 3: Discover many pulsars

For a data release, use layout discovery + file discovery instead of building
`file_data` by hand. Pulsars are matched **by sky position**, not filename.

In [ ]:
epta_layout = discover_layout("../../data/ipta-dr2/EPTA_v2.2", name="EPTA dr2")
ppta_layout = discover_layout("../../data/ipta-dr2/PPTA_dr1dr2", name="PPTA dr1dr2")
nanograv_layout = discover_layout("../../data/ipta-dr2/NANOGrav_9y", name="NANOGrav 9y")
# NG9 ships two J1713 pars; the packaged spec ranks the .t2 solution first.
nanograv_layout["NANOGrav 9y"]["par_precedence"] = PTA_DATA_RELEASES["nanograv_9y"][
    "par_precedence"
]
combined_layout = combine_layouts(epta_layout, ppta_layout, nanograv_layout)

In [ ]:
PTA_DATA_RELEASES.keys(), PTA_DATA_RELEASES["epta_dr2"]

Those layout dicts feed `discover_files`. A discovered layout has no
precedence of its own: if a release ships two pars for one pulsar (NANOGrav 9y
J1713), copy `par_precedence` from `PTA_DATA_RELEASES` rather than picking
silently.

In [ ]:
file_data = discover_files(combined_layout)

In [ ]:
pta_summary(file_data)

`file_data` is still just paths. Names are resolved from par-file coordinates
(B-preferred when a release uses a B name). `J1939+2134` and `B1937+21` are
the same pulsar.

In [ ]:
pulsar_names = get_pulsar_names_from_file_data(file_data)
print(len(pulsar_names), pulsar_names[:5])

pulsar_selection = ["B1855+09", "J1939+2134", "J0030+0451"]
filtered_data = filter_file_data_by_pulsars(file_data, pulsar_selection)
filtered_data

## Part 4: `create_all_metapulsars`

`reference_pta=None` picks the longest timespan per pulsar. Other factory
defaults apply (`shared`, default `AlignmentPolicy()`, `canonicalize_tim=False`).
This batch does **not** write combination products.

In [ ]:
metapulsars = create_all_metapulsars(filtered_data, reference_pta=None)
list(metapulsars)

In [ ]:
psr = metapulsars["B1855+09"]
print(psr.name, len(psr.toas), psr.combination_strategy)